# S1 Backtest — OOS tearsheet

Set **`N_STAR`**, **`TIMING_STAR`**, **`WEIGHT_STAR`**, **`INV_VOL_WINDOW_STAR`** (if `inv_vol`),
**`STOP_STAR`**, **`VT_STAR`**, and **`IC_STAR`** by hand from notebooks 01–07. No further search.

Run the full sample and report **IS** and **OOS** (post-IS) statistics separately —
QuantConnect-style review with emphasis on OOS, plus underperformance diagnostics
(losing months, regimes, IC stability/decay, turnover drag, weight concentration).

Signal: 05 slim-ffill Ridge. Costs: Alpaca-representative open/close one-way bps.


## 0. Imports & Config


In [1]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s1_equities.costs import DEFAULT_COSTS, cost_summary
from backtest.s1_equities.portfolio import (
    WEIGHT_EQUAL,
    WEIGHT_INV_VOL,
    WEIGHT_RANK,
    WEIGHT_SCORE,
)
from backtest.s1_equities.report import (
    compare_segments,
    concentration_vs_returns,
    ic_horizon_decay,
    ic_stability_summary,
    losing_months_summary,
    plot_concentration,
    plot_cost_drag_vs_turnover,
    plot_drawdown,
    plot_equity_overlay,
    plot_losing_months,
    plot_rolling_ic,
    plot_rolling_sharpe,
    regime_performance,
    split_period_returns,
    spy_weekly_open_returns,
    turnover_drag_summary,
    write_tearsheet_pdf,
)
from backtest.s1_equities.runner import run_backtest, summarize_periods
from backtest.s1_equities.stops import parse_stop_star
from backtest.s1_equities.signals import (
    TIMING_MON_OPEN_FRI_CLOSE,
    TIMING_MON_OPEN_MON_OPEN,
    default_backtest_paths,
    is_mask_from_preds,
    load_ohlc_panels,
    load_predictions,
    prediction_path_for_timing,
    score_matrix,
)
from data.processing.feature_implementation.realized_vol import DEFAULT_OPEN_VOL_WINDOW
from models.s1_equities.training_common import (
    LABEL_COL,
    LABEL_COL_MON_FRI,
    date_ic_series,
)
from risk.signal_conviction import parse_ic_scale_star
from risk.vol_targeting import parse_vol_target_star

PATHS = default_backtest_paths(ROOT)
ART = PATHS["artifacts"]
os.makedirs(ART, exist_ok=True)

# --- Set from notebooks 01 / 02 / 03 / 05 / 06 / 07 (suggested values printed there) ---
N_STAR = 15
TIMING_STAR = TIMING_MON_OPEN_MON_OPEN  # or TIMING_MON_OPEN_FRI_CLOSE
WEIGHT_STAR = WEIGHT_INV_VOL  # or WEIGHT_SCORE / WEIGHT_RANK / WEIGHT_INV_VOL
# From notebook 03 §5 when WEIGHT_STAR is inv_vol (ignored otherwise)
INV_VOL_WINDOW_STAR = 42
STOP_STAR = "pct_20"  # "none" | "atr_14_3" from 05 | "pct_5" from 06
VT_STAR = "none"  # from notebook 07, e.g. "vt_bayes_0.94_10_q0.5" or "..._db0.05"
IC_STAR = "none"  # from notebook 07, e.g. "ic_k25_0.94_f0.25_c1.25"

PRED_PATH = prediction_path_for_timing(PATHS, TIMING_STAR)

print(f"N_STAR={N_STAR} (notebook parameter)")
print(f"TIMING_STAR={TIMING_STAR} (notebook parameter)")
print(f"WEIGHT_STAR={WEIGHT_STAR} (notebook parameter)")
print(f"INV_VOL_WINDOW_STAR={INV_VOL_WINDOW_STAR} (used when WEIGHT_STAR=inv_vol)")
print(f"STOP_STAR={STOP_STAR} (notebook parameter)")
print(f"VT_STAR={VT_STAR} (notebook parameter)")
print(f"IC_STAR={IC_STAR} (notebook parameter)")
print(f"predictions={PRED_PATH}")
STOP_CFG = parse_stop_star(STOP_STAR)
VT_CFG = parse_vol_target_star(VT_STAR)
IC_CFG = parse_ic_scale_star(IC_STAR)
display(pd.Series(cost_summary(DEFAULT_COSTS), name="bps").to_frame())


N_STAR=15 (notebook parameter)
TIMING_STAR=mon_open_mon_open (notebook parameter)
WEIGHT_STAR=inv_vol (notebook parameter)
INV_VOL_WINDOW_STAR=42 (used when WEIGHT_STAR=inv_vol)
STOP_STAR=pct_20 (notebook parameter)
predictions=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_artifacts\s1_linear_slim_ffill_is_predictions.parquet


,bps
commission_bps,0.0
regulatory_bps,0.5
half_spread_bps,5.0
open_slippage_bps,10.0
close_slippage_bps,5.0
short_borrow_bps,0.0
open_one_way_bps,15.5
close_one_way_bps,10.5


## 1. Load & run full-sample backtest


In [2]:
preds = load_predictions(PRED_PATH)
scores = score_matrix(preds)
is_mask = is_mask_from_preds(preds)
opens, highs, lows, closes = load_ohlc_panels(PATHS["features"], tickers=list(scores.columns))

label_col = LABEL_COL_MON_FRI if TIMING_STAR == TIMING_MON_OPEN_FRI_CLOSE else LABEL_COL
ic_inputs = None
if IC_CFG.enabled:
    if label_col not in preds.columns:
        raise ValueError(f"predictions missing {label_col} for IC overlay")
    ics = date_ic_series(preds, "score", label_col=label_col)
    n_names = preds.groupby("date")["ticker"].nunique()
    n_names.index = pd.to_datetime(n_names.index)
    ic_inputs = pd.DataFrame({"ic": ics, "n_names": n_names}).sort_index()

result = run_backtest(
    scores,
    opens,
    closes,
    n=N_STAR,
    timing_mode=TIMING_STAR,
    weight_mode=WEIGHT_STAR,
    inv_vol_window=INV_VOL_WINDOW_STAR,
    costs=DEFAULT_COSTS,
    date_mask=None,  # full sample; split in reporting
    highs=highs,
    lows=lows,
    stop=STOP_CFG,
    vol_target=VT_CFG,
    ic_scale=IC_CFG,
    ic_inputs=ic_inputs,
)
print(
    f"periods={len(result.period_returns)}  "
    f"span={result.period_returns.index.min().date()} -> {result.period_returns.index.max().date()}"
)


periods=591  span=2015-03-23 -> 2026-07-13


## 2. IS vs OOS metrics


In [3]:
is_dates = is_mask.index[is_mask]
metrics = compare_segments(result, is_dates)
display(metrics)

# Explicit OOS-only summary line
oos = metrics.loc["OOS"]
print(
    f"OOS: Sharpe={oos['sharpe']:.3f}  AvgAnn={oos['avg_ann_return']:.3%}  "
    f"CAGR={oos['cagr']:.3%}  MaxDD={oos['max_drawdown']:.3%}  "
    f"periods={int(oos['n_periods'])}"
)
is_row = metrics.loc["IS"]
print(
    f"IS:  Sharpe={is_row['sharpe']:.3f}  AvgAnn={is_row['avg_ann_return']:.3%}  "
    f"CAGR={is_row['cagr']:.3%}  MaxDD={is_row['max_drawdown']:.3%}  "
    f"periods={int(is_row['n_periods'])}"
)


,n_periods,total_return,avg_ann_return,cagr,vol,sharpe,max_drawdown,win_rate,mean_return,sharpe_vbt,sortino_vbt,calmar_vbt,max_dd_vbt
segment,,,,,,,,,,,,,
IS,411,0.738380,0.073128,0.072465,0.078988,0.925807,-0.207560,0.600973,0.001406,0.925807,1.335200,0.349129,-0.207560
OOS,180,0.365124,0.094728,0.094082,0.097807,0.968526,-0.149052,0.561111,0.001822,0.968526,1.589350,0.631201,-0.149052
FULL,591,1.373103,0.079707,0.079003,0.085087,0.936763,-0.207560,0.588832,0.001533,0.936763,1.416125,0.380628,-0.207560


OOS: Sharpe=0.969  AvgAnn=9.473%  CAGR=9.408%  MaxDD=-14.905%  periods=180
IS:  Sharpe=0.926  AvgAnn=7.313%  CAGR=7.247%  MaxDD=-20.756%  periods=411


## 3. Performance overview

Headline equity / risk visuals only. Deep-dive diagnostics are in section 4.

### 3.1 Equity (net vs gross)


In [ ]:
is_end = is_dates.max()
_, oos_rets = split_period_returns(result.period_returns, is_dates)

fig = plot_equity_overlay(
    {"net": result.equity, "gross": result.equity_gross},
    title=f"Equity — N={N_STAR}  {TIMING_STAR}  {WEIGHT_STAR}",
    is_end=is_end,
)
plt.show()


### 3.2 Drawdown

In [ ]:
fig = plot_drawdown(result.equity, title="Net drawdown (full sample)")
plt.show()


### 3.3 OOS rolling Sharpe

In [ ]:
fig = plot_rolling_sharpe(
    oos_rets,
    window=26,
    title="Out-of-sample (OOS) rolling Sharpe (26 weeks)",
)
plt.show()


## 4. Underperformance diagnostics

Descriptive OOS diagnostics only — do not retune parameters on this slice.


### 4.1 Losing months

Compound weekly OOS returns to calendar months; list worst months and yearly hit-rate.


In [ ]:
lm = losing_months_summary(oos_rets)
display(lm["worst"])
display(lm["pct_months_up_by_year"].to_frame())
fig = plot_losing_months(lm["monthly"], title="OOS monthly returns")
plt.show()


### 4.2 Regimes

SPY open-to-open weekly return (Up/Down) and trailing-26w SPY vol terciles.
Vol cutpoints are fit on **IS** trailing vol only, then applied to OOS.


In [ ]:
spy_w = spy_weekly_open_returns(result.period_returns.index)
regimes = regime_performance(oos_rets, spy_w, is_dates)
print("Vol cutpoints (IS):")
display(regimes["vol_cutpoints"].to_frame("value"))
print("By market direction:")
display(regimes["market"])
print("By vol tercile:")
display(regimes["vol"])


### 4.3 IC stability and decay

Per-date Spearman IC of score vs the timing label; rolling mean IC; horizon decay.


In [ ]:
LABEL_STAR = (
    LABEL_COL_MON_FRI
    if TIMING_STAR == TIMING_MON_OPEN_FRI_CLOSE
    else LABEL_COL
)
ic = ic_stability_summary(preds, is_dates, label_col=LABEL_STAR)
display(ic["segment"])
fig = plot_rolling_ic(ic["ics_oos"], title="OOS rolling mean IC (26 weeks)")
plt.show()
decay = ic_horizon_decay(preds, PATHS["features"])
display(decay)


### 4.4 Turnover drag

Gross vs net edge and cost drag versus one-way turnover.


In [ ]:
drag = turnover_drag_summary(result, is_dates)
display(drag["summary"].to_frame("value"))
display(drag["corr"])
print(
    f"Mean turnover={drag['summary']['mean_turnover']:.4f}  "
    f"Median={drag['summary']['median_turnover']:.4f}"
)
fig = plot_cost_drag_vs_turnover(drag["frame"])
plt.show()


### 4.5 Weight concentration

Per-date max |w| / HHI on OOS entry weights; correlation with period returns.


In [ ]:
conc = concentration_vs_returns(
    result.entry_weights.reindex(oos_rets.index),
    oos_rets,
)
display(conc["summary"].to_frame("value"))
display(conc["corr"])
fig = plot_concentration(conc["stats"], title="OOS weight concentration")
plt.show()


## 5. Save tearsheet PDF

Exports metrics, performance overview, and underperformance diagnostics.


In [ ]:
pdf_path = os.path.join(
    ART, f"tearsheet_n{N_STAR}_{TIMING_STAR}_{WEIGHT_STAR}.pdf"
)
LABEL_STAR = (
    LABEL_COL_MON_FRI
    if TIMING_STAR == TIMING_MON_OPEN_FRI_CLOSE
    else LABEL_COL
)
write_tearsheet_pdf(
    pdf_path,
    result=result,
    is_dates=is_dates,
    metrics=metrics,
    title=f"S1 L/S N={N_STAR} {TIMING_STAR} {WEIGHT_STAR}",
    preds=preds,
    features_path=PATHS["features"],
    label_col=LABEL_STAR,
)
print(f"saved {pdf_path}")
